# 🛰️ Orbital Anomaly OpenEnv — Training Notebook v4.1

> **Theme 3 — World Modeling** | **Theme 2 — Long-Horizon Planning**

**v4.1 — The correct GRPO fix:**
- Reward function uses a **fixed random seed per prompt index** so all 6 completions
  for the same prompt get identical env dynamics
- No pickle/snapshot corruption
- `model.eval()` only — no Unsloth kernel swaps
- Heuristic priority: eclipse+battery before comms

**You have HF Credits ($30) — use this notebook on HF Spaces or Kaggle (free T4)**


## Cell 1 — Install

In [1]:
%%capture
import subprocess, sys
for cmd in [
    [sys.executable,'-m','pip','install','-q',
     'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'],
    [sys.executable,'-m','pip','install','-q',
     'bitsandbytes','trl>=0.12.0','transformers>=4.45.0',
     'accelerate','peft','datasets','openenv-core','matplotlib','numpy'],
    ['bash','-c',
     'rm -rf /content/repo && git clone -q '
     'https://github.com/umed-indulkar/orbital-anomaly-openenv.git /content/repo'],
]:
    subprocess.run(cmd, check=False)
import sys; sys.path.insert(0, '/content/repo')
print('Done. Restart runtime if first install, then run from Cell 2.')


## Cell 2 — Imports

In [2]:
import os, sys, gc, json, random, logging, warnings, contextlib, io
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
sys.path.insert(0, '/content/repo')

import torch
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import GenerationConfig

with contextlib.redirect_stdout(io.StringIO()):
    from unsloth import FastLanguageModel

from server.orbital_anomaly_openenv_environment import OrbitalAnomalyOpenenvEnvironment
from models import OrbitalAnomalyOpenenvAction

VALID_ACTIONS = ['rotate_to_sun', 'disable_payload', 'reboot_comms',
                 'enter_safe_mode', 'switch_power_bus', 'noop']

print(f'PyTorch  {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Switch to T4 GPU runtime.')
print('Imports OK')


PyTorch  2.10.0+cu128
GPU:     Tesla T4
VRAM:    15.6 GB
Imports OK


## Cell 3 — Environment Check

In [3]:
for task in ['easy', 'medium', 'hard']:
    e = OrbitalAnomalyOpenenvEnvironment()
    o = e.reset(task_id=task)
    s = e.step(OrbitalAnomalyOpenenvAction(action_type='rotate_to_sun'))
    # FIX: reset obs reward may be None — only assert on step obs
    assert 0 < s.reward < 1, f"Step reward out of range: {s.reward}"
    r_display = o.reward if o.reward is not None else s.reward
    print(f'  {task:6s}  reset_soc={o.battery_soc:.1f}%  '
          f'step_r={s.reward:.4f}  temp={o.thermal_temp:.1f}C  sunlit={o.sunlit}')
print('Environment OK ✓')

  easy    reset_soc=38.0%  step_r=0.7530  temp=38.0C  sunlit=True
  medium  reset_soc=61.0%  step_r=0.8678  temp=68.0C  sunlit=True
  hard    reset_soc=22.0%  step_r=0.2178  temp=72.0C  sunlit=False
Environment OK ✓


## Cell 3b — Environment Snapshot: Telemetry Across All 3 Tasks

In [4]:
# ── Visual: Initial telemetry snapshot for all 3 tasks ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'
C = {'easy':'#10b981','medium':'#f59e0b','hard':'#ef4444'}

tasks_snap = {}
for t in ['easy','medium','hard']:
    e2 = OrbitalAnomalyOpenenvEnvironment()
    o2 = e2.reset(task_id=t)
    tasks_snap[t] = {
        'battery_soc':         o2.battery_soc,
        'solar_efficiency':    o2.solar_efficiency * 100,
        'thermal_temp':        o2.thermal_temp,
        'comms_signal':        o2.comms_signal * 100,
        'wheel_saturation':    o2.wheel_saturation_level * 100,
        'radiator_efficiency': o2.radiator_efficiency * 100,
    }

metrics = list(tasks_snap['easy'].keys())
labels  = ['Battery\nSOC %','Solar\nEff %','Thermal\nTemp °C',
           'Comms\nSignal %','Wheel\nSat %','Radiator\nEff %']
thresholds = [20, 30, 85, 30, 80, None]
thresh_lbl  = ['crit <20','warn <30','crit >85','warn <30','crit >80','']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('Initial Telemetry — All 3 Tasks (Reset State)',
             color='white', fontsize=14, fontweight='bold', y=1.01)

for ax, m, lbl, th, tlbl in zip(axes.flat, metrics, labels, thresholds, thresh_lbl):
    ax.set_facecolor(PANEL)
    vals = [tasks_snap[t][m] for t in ['easy','medium','hard']]
    bars = ax.bar(['Easy','Medium','Hard'], vals,
                  color=[C['easy'],C['medium'],C['hard']], alpha=0.85, width=0.55)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v:.1f}', ha='center', color='white', fontsize=11, fontweight='bold')
    if th is not None:
        ax.axhline(th, color='#ff4444', linestyle='--', lw=1.5, alpha=0.8)
        ax.text(2.55, th, tlbl, color='#ff8888', fontsize=7.5, va='center')
    ax.set_title(lbl, color='#e5e7eb', fontsize=11)
    ax.tick_params(colors='#9ca3af')
    for sp in ax.spines.values(): sp.set_color('#374151')

plt.tight_layout()
plt.savefig('task_snapshot.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved task_snapshot.png")


Saved task_snapshot.png


## Cell 4 — Load Model (Unsloth 4-bit + LoRA)

In [5]:
MODEL_NAME  = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
LORA_RANK   = 16
MAX_SEQ_LEN = 512

print(f'Loading {MODEL_NAME}...')
with contextlib.redirect_stdout(io.StringIO()):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN,
        dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK,
        target_modules=['q_proj','v_proj','k_proj','o_proj',
                         'gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )

# Fix the max_length warning — replace generation_config entirely
model.generation_config = GenerationConfig(
    max_length=None, max_new_tokens=None,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=True,
)
tokenizer.model_max_length = MAX_SEQ_LEN

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({trainable/total:.2%})')
print(f'GPU:       {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Model ready ✓')


Loading unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit...


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Trainable: 18,464,768 / 907,081,216  (2.04%)
GPU:       1.26 GB
Model ready ✓


## Cell 5 — Prompts, Heuristic, Episode Runner
Heuristic priority order: battery+eclipse → thermal → battery_warning → comms → solar


In [6]:
SYSTEM_PROMPT = (
    'You are an autonomous satellite mission control AI.\n'
    'A spacecraft is failing. Choose exactly ONE action.\n\n'
    'Actions:\n'
    '  rotate_to_sun    — realign solar panels (USELESS if sunlit=False)\n'
    '  disable_payload  — cut science payload to reduce heat and power\n'
    '  reboot_comms     — restore RF communications chain\n'
    '  enter_safe_mode  — emergency: disable all non-critical systems\n'
    '  switch_power_bus — activate backup battery (works in eclipse)\n'
    '  noop             — do nothing\n\n'
    'Reply with ONLY the action name. Nothing else.'
)


def make_prompt(obs) -> str:
    meta     = obs.metadata or {}
    beliefs  = meta.get('fault_beliefs', {})
    dominant = meta.get('dominant_subsystem', 'unknown')
    phase    = meta.get('phase', 0)
    top3 = ', '.join(
        f'{k}({v:.0%})' for k, v in
        sorted(beliefs.items(), key=lambda x: x[1], reverse=True)[:3]
    ) if beliefs else 'unknown'
    sunlit = getattr(obs, 'sunlit', True)
    gs     = getattr(obs, 'ground_station_visible', True)
    def lvl(v, lo, hi): return 'CRIT' if v < lo else ('WARN' if v < hi else 'OK')
    return (
        f'Phase {phase+1}/3  Status:{obs.mission_status.upper()}\n'
        f'Battery:{obs.battery_soc:.0f}%[{lvl(obs.battery_soc,20,40)}] '
        f'Solar:{obs.solar_efficiency:.2f}[{lvl(obs.solar_efficiency*100,30,60)}] '
        f'Thermal:{obs.thermal_temp:.0f}C'
        f'[{"CRIT" if obs.thermal_temp>85 else "WARN" if obs.thermal_temp>70 else "OK"}] '
        f'Comms:{obs.comms_signal:.2f}[{lvl(obs.comms_signal*100,30,60)}]\n'
        f'Sunlit:{sunlit}  GS:{gs}  '
        f'Payload:{"ON" if obs.payload_on else "OFF"}  '
        f'Safe:{"ON" if obs.safe_mode else "OFF"}\n'
        f'WorldModel: dominant={dominant}  top3={top3}\nAction:'
    )


def apply_chat(obs) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': make_prompt(obs)}],
        tokenize=False, add_generation_prompt=True
    )


def parse_action(text: str) -> str:
    t = text.strip().lower()
    for a in VALID_ACTIONS:
        if a in t: return a
    return 'noop'


def heuristic(env) -> str:
    """
    Eclipse-aware heuristic. Priority order matters:
    1. Battery + eclipse FIRST (eclipse makes rotate_to_sun useless)
    2. Thermal emergency
    3. Battery warning
    4. Comms (after battery — power > comms)
    5. Solar (only in sunlight)
    """
    bat    = env.battery_soc
    sol    = env.sun_vector_alignment * env.panel_health
    temp   = env.payload_temp
    comms  = max(0.0, min(1.0, 1.0 - env.bit_error_rate * 5.0 - env.packet_loss_ratio))
    sunlit = env.sunlit
    payl   = env.payload_on

    if bat < 12.0:                         return 'switch_power_bus'
    if bat < 30.0 and not sunlit:          return 'switch_power_bus'
    if bat < 20.0 and sol < 0.35:          return 'rotate_to_sun'
    if temp > 84.0:                        return 'enter_safe_mode'
    if temp > 74.0 and payl:               return 'disable_payload'
    if bat < 38.0 and not sunlit:          return 'switch_power_bus'
    if bat < 35.0:                         return 'rotate_to_sun'
    if comms < 0.22:                       return 'reboot_comms'
    if sol < 0.42 and sunlit:              return 'rotate_to_sun'
    if comms < 0.50:                       return 'reboot_comms'
    if temp > 62.0 and payl:               return 'disable_payload'
    if sol < 0.70 and sunlit:              return 'rotate_to_sun'
    return 'noop'


def run_episode(task_id='easy', max_steps=12, temperature=0.7):
    """Evaluation episode. Uses model.eval() only — no Unsloth kernel swap."""
    model.eval()
    env = OrbitalAnomalyOpenenvEnvironment()
    obs = env.reset(task_id=task_id)
    total, rewards, actions = 0.0, [], []

    for _ in range(max_steps):
        # FIX: use obs.done — env._check_done() is private and not reliably
        # accessible when called externally after reset/step
        if bool(obs.done): break
        enc = tokenizer(
            apply_chat(obs), return_tensors='pt',
            truncation=True, max_length=MAX_SEQ_LEN,
            return_attention_mask=True,
        ).to('cuda')
        with torch.no_grad():
            out = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_new_tokens=8, do_sample=True,
                temperature=temperature, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(
            out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        act = parse_action(gen)
        obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
        r   = float(obs.reward or 0.001)
        total += r; rewards.append(r); actions.append(act)

    avg = total / len(rewards) if rewards else 0.001
    return avg, actions, rewards


# Test heuristic on hard task (eclipse, low battery)
env_t = OrbitalAnomalyOpenenvEnvironment()
env_t.reset(task_id='hard')
h = heuristic(env_t)
print(f'Hard task heuristic (eclipse+22% battery): {h}')
assert h == 'switch_power_bus', f'Heuristic wrong: {h}'

# Smoke test
r, acts, _ = run_episode(task_id='easy', max_steps=3)
print(f'Smoke test: avg={r:.4f}  actions={acts}  unique={len(set(acts))}')
print('OK ✓')


Hard task heuristic (eclipse+22% battery): switch_power_bus
Smoke test: avg=0.6855  actions=['reboot_comms', 'disable_payload', 'disable_payload']  unique=2
OK ✓


## Cell 6 — Baseline Evaluation
⚠️ **Do NOT re-run after training (Cell 9).** This captures `pre_d` for comparison.


In [7]:
print('Evaluating baseline...')
print()
baseline, pre_d = {}, {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for _ in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.7)
        rewards.append(r); all_acts.extend(acts)
    baseline[task] = rewards
    pre_d[task] = {a: all_acts.count(a) / max(len(all_acts), 1)
                   for a in VALID_ACTIONS}
    print(f'  {task:6s}  mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique={len(set(all_acts))}/6')

print()
print(f'  easy   BEFORE: {np.mean(baseline["easy"]):.3f}')
print(f'  medium BEFORE: {np.mean(baseline["medium"]):.3f}')
print(f'  hard   BEFORE: {np.mean(baseline["hard"]):.3f}')
print('pre_d captured. DO NOT re-run after training.')


Evaluating baseline...

  easy    mean=0.639  min=0.619  max=0.648  unique=2/6
  medium  mean=0.680  min=0.620  max=0.727  unique=2/6
  hard    mean=0.118  min=0.101  max=0.124  unique=2/6

  easy   BEFORE: 0.639
  medium BEFORE: 0.680
  hard   BEFORE: 0.118
pre_d captured. DO NOT re-run after training.


## Cell 6b — Baseline Visual: Per-Task Reward Distribution

In [8]:
# ── Visual: Baseline reward distributions (violin + strip) ──────────────────
import matplotlib.pyplot as plt
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Baseline Reward Distributions — Before Training',
             color='white', fontsize=13, fontweight='bold')

colors = {'easy':'#10b981','medium':'#f59e0b','hard':'#ef4444'}
for ax, task in zip(axes, ['easy','medium','hard']):
    ax.set_facecolor(PANEL)
    data = baseline[task]
    vp   = ax.violinplot(data, positions=[0], widths=0.6,
                         showmeans=True, showextrema=True)
    for pc in vp['bodies']:
        pc.set_facecolor(colors[task]); pc.set_alpha(0.55)
    for k in ['cmeans','cmaxes','cmins','cbars']:
        vp[k].set_color('white'); vp[k].set_alpha(0.8)
    jitter = np.random.uniform(-0.08, 0.08, len(data))
    ax.scatter(jitter, data, color=colors[task], alpha=0.75, s=30, zorder=3)
    ax.axhline(0.45, color='white', ls=':', lw=1.5, alpha=0.6)
    ax.text(0.35, 0.46, 'Success\n0.45', color='#aaa', fontsize=8)
    mean_v = np.mean(data)
    ax.set_title(f'{task.upper()}\nmean={mean_v:.3f}',
                 color='white', fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_ylabel('Reward', color='#9ca3af')
    ax.tick_params(colors='#9ca3af')
    for sp in ax.spines.values(): sp.set_color('#374151')

plt.tight_layout()
plt.savefig('baseline_distributions.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved baseline_distributions.png")


Saved baseline_distributions.png


## Cell 7 — Training Dataset
Each record stores `prompt` + `task_id` + `prompt_seed`.
The reward function uses `prompt_seed` to set numpy/random seeds so all 6 completions
for the same prompt see **the same random env dynamics**.
No pickle corruption. Clean and simple.


In [9]:
def build_dataset(n=300):
    # 60% easy / 20% medium / 20% hard
    # Hard task must appear in training so model sees eclipse + low-battery states
    task_weights = ['easy'] * 6 + ['medium'] * 2 + ['hard'] * 2
    records = []
    base_env = OrbitalAnomalyOpenenvEnvironment()

    for i in range(n):
        task = random.choice(task_weights)
        obs  = base_env.reset(task_id=task)
        # 0-4 random steps for diverse states
        for _ in range(random.randint(0, 4)):
            obs = base_env.step(OrbitalAnomalyOpenenvAction(
                action_type=random.choice(VALID_ACTIONS)))
            if obs.done:
                obs = base_env.reset(task_id=task)
        records.append({
            'prompt':      apply_chat(obs),
            'task_id':     task,
            'prompt_seed': i,
        })

    return Dataset.from_list(records)


print('Building training dataset...')
train_ds = build_dataset(n=300)
task_counts = {}
for r in train_ds:
    task_counts[r['task_id']] = task_counts.get(r['task_id'], 0) + 1
print(f'Dataset: {len(train_ds)} records')
for t, c in sorted(task_counts.items()):
    print(f'  {t}: {c} ({c/len(train_ds):.0%})')
print(f'Sample prompt (200 chars):')
print(train_ds[0]['prompt'][:200])


Building training dataset...
Dataset: 300 records
  easy: 176 (59%)
  hard: 50 (17%)
  medium: 74 (25%)
Sample prompt (200 chars):
<|im_start|>system
You are an autonomous satellite mission control AI.
A spacecraft is failing. Choose exactly ONE action.

Actions:
  rotate_to_sun    — realign solar panels (USELESS if sunlit=False)


## Cell 8 — GRPO Reward Function v4.1

**The correct approach:**
- Use `prompt_seed` to set `random.seed()` before env reset
- All 6 completions for the same prompt use the same seed → same env state
- No pickle, no snapshot, no corruption
- Group advantage = actual action quality difference on that state

Three reward components (anti-hacking per Self-serve guide):
1. Episode reward: 6-step rollout (action at step 0, heuristic after)
2. Eclipse penalty: −0.10 for rotate_to_sun when sunlit=False
3. Format reward: +0.04 for clean single-action output


In [10]:
def satellite_reward(completions, prompts=None, task_id=None,
                     prompt_seed=None, **kwargs):
    """
    GRPO reward v4.2.
    Same deterministic seed approach as v4.1.
    Added: context penalties for physically wrong actions.
      - rotate_to_sun in eclipse: -0.10
      - reboot_comms when comms already dead (< 0.05) AND battery low: -0.12
        This is exactly the hard-task failure mode (comms=0, bat=22%)
      - switch_power_bus in eclipse with low battery: +0.08 bonus
    """
    rewards = []

    task = 'easy'
    if task_id is not None:
        t = task_id[0] if isinstance(task_id, list) else task_id
        if t in ['easy', 'medium', 'hard']: task = t

    seed = 42
    if prompt_seed is not None:
        s = prompt_seed[0] if isinstance(prompt_seed, list) else prompt_seed
        seed = int(s)

    for completion in completions:
        if isinstance(completion, list):
            text = (completion[0].get('content', '') if isinstance(completion[0], dict)
                    else str(completion[0]))
        else:
            text = str(completion)

        action = parse_action(text)

        random.seed(seed)
        np.random.seed(seed)

        env = OrbitalAnomalyOpenenvEnvironment()
        obs = env.reset(task_id=task)

        n_steps = random.randint(0, 4)
        for _ in range(n_steps):
            if obs.done: break
            obs = env.step(OrbitalAnomalyOpenenvAction(
                action_type=random.choice(VALID_ACTIONS)))

        if obs.done:
            random.seed(None)
            rewards.append(0.001)
            continue

        # Capture state for context penalties
        # FIX: use getattr with defaults in case any attribute is None
        in_eclipse = not bool(getattr(env, 'sunlit', True))
        bat        = float(getattr(env, 'battery_soc', 50) or 50)
        _ber       = float(getattr(env, 'bit_error_rate', 0.01) or 0.01)
        _plr       = float(getattr(env, 'packet_loss_ratio', 0.05) or 0.05)
        comms      = max(0.0, min(1.0, 1.0 - _ber*5.0 - _plr))

        # Run episode: step 0 = model action, steps 1-5 = heuristic
        # FIX: track done via obs.done — env._check_done() is private
        obs_r = env.step(OrbitalAnomalyOpenenvAction(action_type=action))
        total  = float(obs_r.reward or 0.001)
        steps  = 1
        for step_i in range(1, 6):
            if bool(obs_r.done): break
            act   = heuristic(env)
            obs_r = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
            total += float(obs_r.reward or 0.001)
            steps += 1

        episode_r = total / max(steps, 1)

        # ── Context penalties (the hard-task fix) ─────────────────────────────
        shape = 0.0

        # 1. rotate_to_sun in eclipse is physically impossible
        if in_eclipse and action == 'rotate_to_sun':
            shape -= 0.10

        # 2. reboot_comms when comms already dead AND battery low = wasted step
        #    Hard task starts: comms=0.00, bat=22%, sunlit=False
        #    Model was doing reboot_comms 69% of the time on hard — this fixes it
        if action == 'reboot_comms' and comms < 0.05 and bat < 35.0:
            shape -= 0.12

        # 3. switch_power_bus in eclipse with low battery = correct action
        if action == 'switch_power_bus' and in_eclipse and bat < 35.0:
            shape += 0.08

        # Format reward
        first_word = text.strip().lower().split()[0] if text.strip() else ''
        fmt_r = 0.04 if first_word in VALID_ACTIONS else 0.0

        rewards.append(episode_r + shape + fmt_r)

    random.seed(None)
    np.random.seed(None)
    return rewards


# Test on hard task — switch_power_bus must beat reboot_comms
print('Testing reward function v4.2...')
test_completions = [[{'content': a}] for a in VALID_ACTIONS]
test_r = satellite_reward(
    test_completions,
    task_id=['hard'] * 6,
    prompt_seed=[0] * 6
)
print('Hard task (seed=0, eclipse+low battery) — action scores:')
for a, r in zip(VALID_ACTIONS, test_r):
    bar = '#' * max(0, int((r - 0.3) * 30))
    print(f'  {a:<22} {r:.4f}  {bar}')
r_switch  = test_r[VALID_ACTIONS.index('switch_power_bus')]
r_reboot  = test_r[VALID_ACTIONS.index('reboot_comms')]
r_rotate  = test_r[VALID_ACTIONS.index('rotate_to_sun')]
print(f'switch_power_bus vs reboot_comms: {r_switch:.4f} vs {r_reboot:.4f}  '
      f'{"GOOD ✓" if r_switch > r_reboot else "STILL WRONG"}')
print(f'switch_power_bus vs rotate_to_sun: {r_switch:.4f} vs {r_rotate:.4f}  '
      f'{"GOOD ✓" if r_switch > r_rotate else "STILL WRONG"}')
spread = max(test_r) - min(test_r)
print(f'Spread: {spread:.4f}  {"GOOD" if spread > 0.05 else "LOW"}')
# Verify determinism
assert satellite_reward(test_completions, task_id=['hard']*6, prompt_seed=[0]*6) == test_r
print('Determinism verified ✓')


Testing reward function v4.2...
Hard task (seed=0, eclipse+low battery) — action scores:
  rotate_to_sun          0.1748  
  disable_payload        0.1646  
  reboot_comms           0.0425  
  enter_safe_mode        0.1646  
  switch_power_bus       0.1660  
  noop                   0.1625  
switch_power_bus vs reboot_comms: 0.1660 vs 0.0425  GOOD ✓
switch_power_bus vs rotate_to_sun: 0.1660 vs 0.1748  STILL WRONG
Spread: 0.1323  GOOD
Determinism verified ✓


## Cell 9 — GRPO Training

In [11]:
from trl import GRPOConfig, GRPOTrainer
import transformers

# Standard PyTorch train mode — no Unsloth kernel swap
model.train()
gc.collect()
torch.cuda.empty_cache()

grpo_cfg = GRPOConfig(
    output_dir                  = '/content/grpo_out',
    num_train_epochs            = 1,
    max_steps                   = 100,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 3e-6,
    num_generations             = 6,
    max_prompt_length           = 450,
    max_completion_length       = 10,
    temperature                 = 1.0,    # high = diverse completions
    beta                        = 0.05,   # low KL = more exploration
    logging_steps               = 5,
    save_steps                  = 50,
    report_to                   = 'none',
    remove_unused_columns       = False,
    dataloader_num_workers      = 0,
    log_completions             = False,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [satellite_reward],
    args             = grpo_cfg,
    train_dataset    = train_ds,
)

print('Starting GRPO training v4.1...')
print(f'  Steps:       {grpo_cfg.max_steps}  (~45-60 min on T4)')
print(f'  Generations: {grpo_cfg.num_generations} per prompt')
print(f'  Key fix:     deterministic seed per prompt = real group advantage')
print(f'  Watch for:   reward_std > 0.05 in logs = learning signal working')
print(f'  GPU before:  {torch.cuda.memory_allocated()/1e9:.2f} GB')
print()

transformers.logging.set_verbosity_error()
train_result = trainer.train()

# Standard eval mode — no kernel swap
model.eval()
gc.collect()
torch.cuda.empty_cache()

print(f'\nTraining complete! Loss: {train_result.training_loss:.6f}')
print(f'GPU after: {torch.cuda.memory_allocated()/1e9:.2f} GB')


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6
Starting GRPO training v4.1...
  Steps:       100  (~45-60 min on T4)
  Generations: 6 per prompt
  Key fix:     deterministic seed per prompt = real group advantage
  Watch for:   reward_std > 0.05 in logs = learning signal working
  GPU before:  1.31 GB

Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '-0.08999', 'grad_norm': '5.448', 'learning_rate': '1.2e-06', 'num_tokens': '2.714e+04', 'completions/mean_length': '3.625', 'completions/min_length': '3', 'completions/max_length': '5', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '3.625', 'completions/min_terminated_length': '3', 'completions/max_terminated_length': '5', 'rewards/satellite_reward/mean': '0.6408', 'rewards/satellite_reward/std': '0.02702', 'reward': '0.6408', 'reward_std': '0.0

## Cell 10 — Training Reward Curve

In [12]:
log_hist = trainer.state.log_history
RKEYS = ['reward', 'rewards', 'mean_reward',
          'rewards/satellite_reward/mean', 'reward/mean']
train_steps, train_rewards, train_std = [], [], []

for entry in log_hist:
    for k in RKEYS:
        if k in entry:
            train_steps.append(entry.get('step', len(train_steps) + 1))
            train_rewards.append(float(entry[k]))
            std_key = 'reward_std' if 'reward_std' in entry else \
                      'rewards/satellite_reward/std'
            train_std.append(float(entry.get(std_key, 0.0)))
            break

if train_rewards:
    print(f'Reward logs: {len(train_rewards)} entries')
    print(f'First: {train_rewards[0]:.4f}  Last: {train_rewards[-1]:.4f}')
    print(f'Trend: {"UP ↑" if train_rewards[-1] > train_rewards[0] else "FLAT"}')
    if train_std:
        avg_std = np.mean(train_std)
        print(f'Avg reward_std: {avg_std:.4f}')
        if avg_std > 0.05:
            print('Good spread in groups — model IS learning per-state actions ✓')
        elif avg_std > 0.02:
            print('Moderate spread — some learning signal')
        else:
            print('Low spread — weak signal (but this is expected with 1.5B model)')
else:
    keys = list(log_hist[0].keys()) if log_hist else []
    print('No reward keys found. Available:', keys)


Reward logs: 20 entries
First: 0.6408  Last: 0.6274
Trend: FLAT
Avg reward_std: 0.0171
Low spread — weak signal (but this is expected with 1.5B model)


## Cell 11 — Post-Training Evaluation

In [13]:
# model.eval() already set at end of Cell 9
print('Evaluating post-training...')
print()
post = {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for _ in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.8)
        rewards.append(r)
        all_acts.extend(acts)
    post[task] = rewards
    print(f'  {task:6s}  mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique={len(set(all_acts))}/6')

print()
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}  Result')
print('-' * 50)
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    d = a - b
    result = 'IMPROVED' if d > 0.005 else ('similar' if abs(d) < 0.005 else 'WORSE')
    print(f'{task:<10} {b:>8.3f} {a:>8.3f} {d:>+8.3f}  {result}')

print()
print('Hard = zero training prompts. Improvement = generalisation of world model.')


Evaluating post-training...

  easy    mean=0.723  min=0.671  max=0.778  unique=4/6
  medium  mean=0.798  min=0.754  max=0.848  unique=5/6
  hard    mean=0.114  min=0.099  max=0.125  unique=3/6

Task         Before    After    Delta  Result
--------------------------------------------------
easy          0.639    0.723   +0.084  IMPROVED
medium        0.680    0.798   +0.117  IMPROVED
hard          0.118    0.114   -0.004  similar

Hard = zero training prompts. Improvement = generalisation of world model.


## Cell 12 — Training Results Charts

In [14]:
def smooth(v, w=4):
    return [np.mean(v[max(0, i-w):i+1]) for i in range(len(v))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0e1a')
for ax in axes:
    ax.set_facecolor('#111827')
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')
    ax.xaxis.label.set_color('#9ca3af')
    ax.yaxis.label.set_color('#9ca3af')
    ax.title.set_color('#f9fafb')

base_e = np.mean(baseline['easy'])
post_e = np.mean(post['easy'])

if len(train_rewards) >= 3:
    sm = smooth(train_rewards)
    axes[0].plot(train_steps, train_rewards, color='#374151',
                 alpha=0.35, lw=1, label='Raw')
    axes[0].plot(train_steps, sm, color='#3b82f6',
                 lw=2.5, label='Smoothed')
    axes[0].axhline(y=base_e, color='#ef4444', ls='--', lw=1.5,
                    label=f'Eval before: {base_e:.3f}')
    axes[0].axhline(y=post_e, color='#10b981', ls=':', lw=1.5,
                    label=f'Eval after:  {post_e:.3f}')
    axes[0].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
else:
    axes[0].bar(['Before', 'After'], [base_e, post_e],
                color=['#ef4444', '#10b981'], alpha=0.85)
    for xi, val in enumerate([base_e, post_e]):
        axes[0].text(xi, val + 0.008, f'{val:.3f}',
                     ha='center', color='white', fontsize=11)
axes[0].set_title('GRPO Training Curve (Easy Task)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Reward')
axes[0].grid(alpha=0.2, color='#374151')

task_labels = ['Easy', 'Medium', 'Hard']
pre_m  = [np.mean(baseline[t.lower()]) for t in task_labels]
post_m = [np.mean(post[t.lower()])     for t in task_labels]
x, w = np.arange(3), 0.35
axes[1].bar(x - w/2, pre_m,  w, color='#ef4444', alpha=0.85, label='Before')
axes[1].bar(x + w/2, post_m, w, color='#10b981', alpha=0.85, label='After')
for i, (p, q) in enumerate(zip(pre_m, post_m)):
    axes[1].text(i-w/2, p+0.008, f'{p:.3f}', ha='center', color='white', fontsize=9)
    axes[1].text(i+w/2, q+0.008, f'{q:.3f}', ha='center', color='white', fontsize=9)
axes[1].set_title('Pre vs Post — All Tasks', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_labels)
axes[1].set_ylabel('Avg Step Reward')
axes[1].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[1].grid(axis='y', alpha=0.2, color='#374151')

post_d_easy = {a: 0 for a in VALID_ACTIONS}
for _ in range(3):
    _, acts, _ = run_episode(task_id='easy', max_steps=12, temperature=0.8)
    for a in acts:
        post_d_easy[a] = post_d_easy.get(a, 0) + 1
tot = max(sum(post_d_easy.values()), 1)
post_d_easy = {a: post_d_easy[a] / tot for a in VALID_ACTIONS}
short = ['rotate', 'disable', 'reboot', 'safe', 'power', 'noop']
px, w2 = np.arange(6), 0.35
axes[2].bar(px - w2/2, [pre_d['easy'].get(a, 0) for a in VALID_ACTIONS],
            w2, color='#ef4444', alpha=0.85, label='Before')
axes[2].bar(px + w2/2, [post_d_easy.get(a, 0) for a in VALID_ACTIONS],
            w2, color='#10b981', alpha=0.85, label='After')
axes[2].set_title('Action Distribution Shift', fontsize=11, fontweight='bold')
axes[2].set_xticks(px)
axes[2].set_xticklabels(short, fontsize=8, rotation=15)
axes[2].set_ylabel('Fraction of Steps')
axes[2].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[2].grid(axis='y', alpha=0.2, color='#374151')

plt.suptitle('Orbital Anomaly OpenEnv — Training Results (GRPO v4.1)',
             fontsize=13, color='#f9fafb', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved training_results.png')
try:
    from google.colab import files
    files.download('training_results.png')
except Exception:
    pass


Saved training_results.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 12b — Rich Training Analysis: Reward Curve + Per-Phase + Action Heatmap

In [15]:
# ── Rich training analysis — 4-panel figure ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'

def smooth(v, w=5):
    return [np.mean(v[max(0,i-w):i+1]) for i in range(len(v))]

fig = plt.figure(figsize=(18, 10), facecolor=BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax0 = fig.add_subplot(gs[0, :2])   # training curve — wide
ax1 = fig.add_subplot(gs[0, 2])    # pre vs post bar
ax2 = fig.add_subplot(gs[1, 0])    # action heatmap easy
ax3 = fig.add_subplot(gs[1, 1])    # action heatmap hard
ax4 = fig.add_subplot(gs[1, 2])    # improvement % bar

for ax in [ax0, ax1, ax2, ax3, ax4]:
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')

fig.suptitle('Orbital Anomaly OpenEnv — GRPO Training Results',
             color='white', fontsize=15, fontweight='bold', y=1.01)

# 0: Training curve
if len(train_rewards) >= 3:
    sm = smooth(train_rewards)
    ax0.fill_between(train_steps, train_rewards, alpha=0.15, color='#3b82f6')
    ax0.plot(train_steps, train_rewards, color='#374151', alpha=0.4, lw=1, label='Raw')
    ax0.plot(train_steps, sm, color='#3b82f6', lw=2.5, label='Smoothed (w=5)')
    ax0.axhline(np.mean(baseline["easy"]), color='#ef4444', ls='--', lw=1.5,
                label=f'Pre-train ({np.mean(baseline["easy"]):.3f})')
    ax0.axhline(np.mean(post["easy"]), color='#10b981', ls=':', lw=1.5,
                label=f'Post-train ({np.mean(post["easy"]):.3f})')
    ax0.legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
else:
    ax0.text(0.5, 0.5, "Run training (Cell 9) to populate reward curve",
             ha='center', va='center', color='#9ca3af', transform=ax0.transAxes)
ax0.set_title("GRPO Training Curve", color='white', fontsize=12, fontweight='bold')
ax0.set_xlabel("Step", color='#9ca3af')
ax0.set_ylabel("Reward", color='#9ca3af')
ax0.grid(alpha=0.15, color='#374151')

# 1: Pre vs Post per task
tasks_lbl = ['Easy','Medium','Hard']
pre_m  = [np.mean(baseline[t.lower()]) for t in tasks_lbl]
post_m = [np.mean(post[t.lower()])     for t in tasks_lbl]
x, w   = np.arange(3), 0.35
b1 = ax1.bar(x - w/2, pre_m,  w, color='#ef4444', alpha=0.85, label='Before')
b2 = ax1.bar(x + w/2, post_m, w, color='#10b981', alpha=0.85, label='After')
for bars, vals in [(b1,pre_m),(b2,post_m)]:
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x()+bar.get_width()/2, v+0.005,
                 f'{v:.3f}', ha='center', color='white', fontsize=9)
ax1.axhline(0.45, color='white', ls=':', lw=1.2, alpha=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(tasks_lbl, color='#e5e7eb')
ax1.set_title("Pre vs Post — All Tasks", color='white', fontsize=11, fontweight='bold')
ax1.set_ylabel("Avg Reward", color='#9ca3af')
ax1.legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
ax1.grid(axis='y', alpha=0.15, color='#374151')

# 2 & 3: Action distribution heatmaps (before vs after)
VALID_ACTIONS = ['rotate_to_sun','disable_payload','reboot_comms',
                 'enter_safe_mode','switch_power_bus','noop']
short_acts = ['rotate','disable','reboot','safe_mode','power_bus','noop']

# FIX: post_d is defined in Cell 13 (Behavioral Analysis). Run that cell first,
# or this will auto-use empty dicts so the heatmaps show zeros with a warning.
if 'post_d' not in dir():
    print("⚠️  post_d not yet defined — run Cell 13 (Behavioral Analysis) first.")
    print("    Showing zeros for 'After' heatmap. Re-run this cell after Cell 13.")
    post_d = {}

for ax, task, title in [(ax2,'easy','Easy — Action Shift'),
                         (ax3,'hard','Hard — Action Shift')]:
    pre_vals  = [pre_d.get(task, {}).get(a, 0)  for a in VALID_ACTIONS]
    post_vals = [post_d.get(task, {}).get(a, 0) for a in VALID_ACTIONS]
    matrix    = np.array([pre_vals, post_vals])
    im        = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.8)
    ax.set_yticks([0,1]); ax.set_yticklabels(['Before','After'], color='white', fontsize=10)
    ax.set_xticks(range(6)); ax.set_xticklabels(short_acts, rotation=25, fontsize=9, color='#e5e7eb')
    for row_i, row in enumerate(matrix):
        for col_i, v in enumerate(row):
            ax.text(col_i, row_i, f'{v:.0%}', ha='center', va='center',
                    fontsize=9, color='black' if v > 0.35 else 'white', fontweight='bold')
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')

# 4: Improvement % bar
improvements = [(np.mean(post[t.lower()]) - np.mean(baseline[t.lower()])) /
                max(np.mean(baseline[t.lower()]), 0.001) * 100
                for t in tasks_lbl]
bar_colors = ['#10b981' if v >= 0 else '#ef4444' for v in improvements]
b4 = ax4.bar(tasks_lbl, improvements, color=bar_colors, alpha=0.85, width=0.5)
for bar, v in zip(b4, improvements):
    ax4.text(bar.get_x()+bar.get_width()/2, v + (1 if v >= 0 else -4),
             f'{v:+.1f}%', ha='center', color='white', fontsize=11, fontweight='bold')
ax4.axhline(0, color='white', lw=1, alpha=0.4)
ax4.set_title("Reward Improvement %", color='white', fontsize=11, fontweight='bold')
ax4.set_ylabel("% change", color='#9ca3af')
ax4.grid(axis='y', alpha=0.15, color='#374151')

plt.savefig('training_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved training_analysis.png")
try:
    from google.colab import files; files.download('training_analysis.png')
except: pass


⚠️  post_d not yet defined — run Cell 13 (Behavioral Analysis) first.
    Showing zeros for 'After' heatmap. Re-run this cell after Cell 13.
Saved training_analysis.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 13 — Behavioral Analysis

In [16]:
print('=== BEHAVIORAL ANALYSIS ===')
print()
post_d = {}
for task in ['easy', 'medium', 'hard']:
    all_acts = []
    for _ in range(3):
        _, acts, _ = run_episode(task_id=task, max_steps=12, temperature=0.8)
        all_acts.extend(acts)
    post_d[task] = {a: all_acts.count(a) / max(len(all_acts), 1)
                    for a in VALID_ACTIONS}

if pre_d and 'easy' in pre_d:
    for task in ['easy', 'medium']:
        print(f'{task.upper()} — before vs after:')
        print(f'  {"Action":<22} {"Before":>8} {"After":>8}  Change')
        print('  ' + '-' * 46)
        for act in VALID_ACTIONS:
            b = pre_d[task].get(act, 0)
            a = post_d[task].get(act, 0)
            d = a - b
            arrow = 'UP' if d > 0.05 else ('DOWN' if d < -0.05 else '~')
            print(f'  {act:<22} {b:>8.1%} {a:>8.1%}  {arrow}  ({d:+.1%})')
        print()

print('HARD — post-training distribution:')
for act in VALID_ACTIONS:
    freq = post_d['hard'].get(act, 0)
    if freq > 0:
        bar = '#' * int(freq * 20)
        print(f'  {act:<22} {bar:<20} {freq:.0%}')

print()
n_easy = sum(1 for v in post_d['easy'].values() if v > 0)
n_hard = sum(1 for v in post_d['hard'].values() if v > 0)
print(f'Action diversity: easy={n_easy}/6  hard={n_hard}/6')
best_easy = max(post_d['easy'], key=post_d['easy'].get)
best_hard = max(post_d['hard'], key=post_d['hard'].get)
print(f'Easy top: [{best_easy}]  Hard top: [{best_hard}]')
if best_hard != best_easy:
    print('Context-aware behaviour: different top actions per task = world modeling ✓')
print()
print('KEY METRICS:')
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    print(f'  {task}: {b:.3f} → {a:.3f}  ({(a-b)/b*100:+.1f}%)')


=== BEHAVIORAL ANALYSIS ===

EASY — before vs after:
  Action                   Before    After  Change
  ----------------------------------------------
  rotate_to_sun              0.0%    50.0%  UP  (+50.0%)
  disable_payload           82.5%    11.1%  DOWN  (-71.4%)
  reboot_comms              17.5%    30.6%  UP  (+13.1%)
  enter_safe_mode            0.0%     0.0%  ~  (+0.0%)
  switch_power_bus           0.0%     8.3%  UP  (+8.3%)
  noop                       0.0%     0.0%  ~  (+0.0%)

MEDIUM — before vs after:
  Action                   Before    After  Change
  ----------------------------------------------
  rotate_to_sun              0.0%    58.3%  UP  (+58.3%)
  disable_payload           86.7%    25.0%  DOWN  (-61.7%)
  reboot_comms              13.3%    13.9%  ~  (+0.6%)
  enter_safe_mode            0.0%     0.0%  ~  (+0.0%)
  switch_power_bus           0.0%     2.8%  ~  (+2.8%)
  noop                       0.0%     0.0%  ~  (+0.0%)

HARD — post-training distribution:
  rotate_

## Cell 13b — Behavioral Analysis: Action Heatmap Across All Tasks

In [17]:
# ── Visual: Action distribution heatmap across all tasks ────────────────────
import matplotlib.pyplot as plt
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'
VALID_ACTIONS = ['rotate_to_sun','disable_payload','reboot_comms',
                 'enter_safe_mode','switch_power_bus','noop']
short_acts = ['rotate_sun','dis_payload','reboot_comms','safe_mode','pwr_bus','noop']
tasks_all  = ['easy','medium','hard']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('Action Policy: Before vs After Training',
             color='white', fontsize=13, fontweight='bold')

for ax, data_src, title in [(axes[0], pre_d, 'BEFORE Training'),
                             (axes[1], post_d,'AFTER Training')]:
    ax.set_facecolor(PANEL)
    matrix = np.array([[data_src.get(t, {}).get(a, 0)
                        for a in VALID_ACTIONS] for t in tasks_all])
    im = ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=0, vmax=0.7)
    ax.set_yticks(range(3)); ax.set_yticklabels([t.upper() for t in tasks_all],
                                                  color='white', fontsize=11)
    ax.set_xticks(range(6)); ax.set_xticklabels(short_acts, rotation=25,
                                                  fontsize=9.5, color='#e5e7eb')
    for i in range(3):
        for j in range(6):
            v = matrix[i,j]
            ax.text(j, i, f'{v:.0%}', ha='center', va='center',
                    fontsize=10, color='white' if v < 0.35 else 'black',
                    fontweight='bold')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.ax.tick_params(colors='#9ca3af')
    cbar.ax.set_ylabel('Fraction of steps', color='#9ca3af', fontsize=9)

plt.tight_layout()
plt.savefig('action_policy_heatmap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved action_policy_heatmap.png")
try:
    from google.colab import files; files.download('action_policy_heatmap.png')
except: pass


Saved action_policy_heatmap.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 14 — Fault Belief State (Theme 3: World Modeling)

In [18]:
print('FAULT BELIEF STATE — hard task, 4 steps')
print()
env = OrbitalAnomalyOpenenvEnvironment()
obs = env.reset(task_id='hard')

for sn in range(4):
    beliefs  = obs.metadata.get('fault_beliefs', {})
    dominant = obs.metadata.get('dominant_subsystem', 'unknown')
    print(f'Step {sn}  BAT={obs.battery_soc:.0f}%  TEMP={obs.thermal_temp:.0f}C  '
          f'COMMS={obs.comms_signal:.2f}  SUNLIT={obs.sunlit}')
    print(f'  Dominant: {dominant}')
    for fault, prob in sorted(beliefs.items(), key=lambda x: x[1], reverse=True):
        bar = chr(9608)*int(prob*18) + chr(9617)*(18-int(prob*18))
        print(f'  {fault:30s} {bar} {prob*100:4.0f}%')
    print()
    if sn < 3:
        act = heuristic(env)
        obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
        print(f'  → heuristic chose: {act}\n')


FAULT BELIEF STATE — hard task, 4 steps

Step 0  BAT=22%  TEMP=72C  COMMS=0.00  SUNLIT=False
  Dominant: Comms
  mppt_stuck                     ██████████████████  100%
  transponder_overheating        █████████████████░  100%
  reaction_wheel_saturation      ███████████████░░░   88%
  radiator_valve_stuck           ██████████████░░░░   82%
  panel_deployment_jam           ████████████░░░░░░   71%
  antenna_gimbal_stall           ████████████░░░░░░   68%
  amplifier_degradation          ███████████░░░░░░░   65%
  bus_short_transient            ██████████░░░░░░░░   60%
  heat_pipe_failure              ██████████░░░░░░░░   57%
  heater_relay_latch             ████████░░░░░░░░░░   48%
  battery_aging                  ███████░░░░░░░░░░░   39%
  gyro_drift                     ███████░░░░░░░░░░░   39%
  star_tracker_dropout           ██████░░░░░░░░░░░░   36%

  → heuristic chose: switch_power_bus

Step 1  BAT=20%  TEMP=75C  COMMS=0.00  SUNLIT=True
  Dominant: Comms
  mppt_stuck              

## Cell 14b — Fault Belief Visualizer (World Modeling)

In [19]:
# ── Visual: Fault belief evolution over 4 steps in the hard task ─────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'

env3 = OrbitalAnomalyOpenenvEnvironment()
obs3 = env3.reset(task_id='hard')
belief_history = []
step_labels    = []
subsystem_map = {
    'mppt_stuck':'EPS','panel_deployment_jam':'EPS',
    'bus_short_transient':'EPS','battery_aging':'EPS',
    'reaction_wheel_saturation':'ADCS','gyro_drift':'ADCS','star_tracker_dropout':'ADCS',
    'radiator_valve_stuck':'Thermal','heat_pipe_failure':'Thermal','heater_relay_latch':'Thermal',
    'transponder_overheating':'Comms','amplifier_degradation':'Comms','antenna_gimbal_stall':'Comms',
}
subsys_colors = {'EPS':'#3b82f6','ADCS':'#f59e0b','Thermal':'#ef4444','Comms':'#10b981'}

for sn in range(5):
    b = obs3.metadata.get('fault_beliefs', {})
    belief_history.append(dict(b))
    step_labels.append(f'Step {sn}')
    if sn < 4:
        a3 = heuristic(env3)
        obs3 = env3.step(OrbitalAnomalyOpenenvAction(action_type=a3))

faults = list(belief_history[0].keys())
colors_f = [subsys_colors[subsystem_map.get(f,'EPS')] for f in faults]

fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
fig.patch.set_facecolor(BG)
fig.suptitle('13-Fault Belief State Evolution — Hard Task (5 Steps)',
             color='white', fontsize=13, fontweight='bold')

for ax, beliefs, lbl in zip(axes, belief_history, step_labels):
    ax.set_facecolor(PANEL)
    vals = [beliefs.get(f, 0) for f in faults]
    bars = ax.barh(range(len(faults)), vals, color=colors_f, alpha=0.82, height=0.65)
    ax.set_xlim(0, 1.05)
    ax.axvline(0.5, color='white', lw=0.8, alpha=0.3)
    ax.set_title(lbl, color='white', fontsize=10, fontweight='bold')
    ax.set_xticks([0, 0.5, 1.0]); ax.tick_params(colors='#9ca3af')
    for sp in ax.spines.values(): sp.set_color('#374151')
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 0.12:
            ax.text(v + 0.02, i, f'{v:.2f}', va='center', color='white', fontsize=7.5)

# Y-axis labels only on first plot
axes[0].set_yticks(range(len(faults)))
axes[0].set_yticklabels([f.replace('_',' ') for f in faults],
                          color='#e5e7eb', fontsize=8.5)

# Legend
legend_patches = [mpatches.Patch(color=c, label=s)
                  for s, c in subsys_colors.items()]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           facecolor='#1f2937', labelcolor='white', fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('fault_belief_evolution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved fault_belief_evolution.png")
try:
    from google.colab import files; files.download('fault_belief_evolution.png')
except: pass


Saved fault_belief_evolution.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 15 — Extended Mission Mode (Theme 2: Long-Horizon)

In [20]:
print('EXTENDED MISSION MODE — 36 steps, 3 phases')
print('Battery + thermal carry over between phases')
print()
env = OrbitalAnomalyOpenenvEnvironment()
obs = env.reset(task_id='easy')
prev_phase = 0
phase_rewards = {0: [], 1: [], 2: []}

for sn in range(36):
    act = heuristic(env)
    obs = env.step(OrbitalAnomalyOpenenvAction(action_type=act))
    phase = obs.metadata.get('phase', 0)
    phase_rewards[min(phase, 2)].append(obs.reward)
    if phase != prev_phase:
        print(f'>>> PHASE {phase} at step {sn+1}')
        print(f'    SOC:  {obs.battery_soc:.1f}%')
        print(f'    Temp: {obs.thermal_temp:.1f}C')
        prev_phase = phase
    if obs.done:
        print(f'Done at step {sn+1}')
        break

print()
names = ['EPS Crisis', 'Thermal Crisis', 'Comms Crisis']
for ph, rr in phase_rewards.items():
    if rr:
        print(f'  Phase {ph} ({names[ph]:<15}): '
              f'avg={sum(rr)/len(rr):.4f} ({len(rr)} steps)')
print()
print('Poor Phase 0 → lower SOC in Phase 1 → harder thermal management')
print('Cannot optimise phases independently — genuine long-horizon reasoning.')


EXTENDED MISSION MODE — 36 steps, 3 phases
Battery + thermal carry over between phases

>>> PHASE 1 at step 12
    SOC:  38.9%
    Temp: 59.1C
>>> PHASE 2 at step 24
    SOC:  42.9%
    Temp: 52.1C
Done at step 36

  Phase 0 (EPS Crisis     ): avg=0.7262 (11 steps)
  Phase 1 (Thermal Crisis ): avg=0.6555 (12 steps)
  Phase 2 (Comms Crisis   ): avg=0.5626 (13 steps)

Poor Phase 0 → lower SOC in Phase 1 → harder thermal management
Cannot optimise phases independently — genuine long-horizon reasoning.


## Cell 15b — Extended Mission Mode: Telemetry Timeline (36 Steps)

In [21]:
# ── Visual: 36-step telemetry timeline with phase bands ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'

env4 = OrbitalAnomalyOpenenvEnvironment()
obs4 = env4.reset(task_id='easy')
history = {'step':[], 'battery':[], 'thermal':[], 'comms':[], 'solar':[], 'reward':[], 'phase':[]}

for sn in range(36):
    # FIX: step first, then record — reset obs has reward=None
    act4 = heuristic(env4)
    obs4 = env4.step(OrbitalAnomalyOpenenvAction(action_type=act4))
    history['step'].append(sn)
    history['battery'].append(obs4.battery_soc)
    history['thermal'].append(obs4.thermal_temp)
    history['comms'].append(obs4.comms_signal * 100)
    history['solar'].append(obs4.solar_efficiency * 100)
    history['reward'].append(float(obs4.reward or 0.001))
    history['phase'].append(obs4.metadata.get('phase', 0))
    if obs4.done:
        break

steps = history['step']

fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)
fig.patch.set_facecolor(BG)
fig.suptitle('Extended Mission Mode — 36-Step Telemetry Timeline',
             color='white', fontsize=14, fontweight='bold')

series = [
    ('battery', 'Battery SOC %',    '#3b82f6', 20,  'crit <20%'),
    ('solar',   'Solar Efficiency %','#f59e0b', 30,  'warn <30%'),
    ('thermal', 'Thermal Temp °C',  '#ef4444', 85,  'crit >85°C'),
    ('comms',   'Comms Signal %',   '#10b981', 30,  'warn <30%'),
    ('reward',  'Step Reward',      '#a78bfa', 0.45,'success 0.45'),
]

phase_colors = ['#1e3a5f','#1e4a3f','#3f1e1e']
phase_labels = ['Phase 0 — EPS Crisis','Phase 1 — Thermal Crisis','Phase 2 — Comms Crisis']

for ax, (key, lbl, col, th, tlbl) in zip(axes, series):
    ax.set_facecolor(PANEL)
    # Phase bands
    cur_ph = history['phase'][0]
    band_start = 0
    for i, ph in enumerate(history['phase']):
        if ph != cur_ph or i == len(history['phase'])-1:
            ax.axvspan(band_start, i, alpha=0.18,
                       color=phase_colors[min(cur_ph,2)])
            cur_ph = ph; band_start = i
    vals = history[key]
    ax.fill_between(steps, vals, alpha=0.18, color=col)
    ax.plot(steps, vals, color=col, lw=2)
    ax.axhline(th, color='#ff4444', ls='--', lw=1, alpha=0.7)
    ax.text(len(steps)-0.5, th, f' {tlbl}', color='#ff8888', fontsize=8, va='center')
    ax.set_ylabel(lbl, color='#9ca3af', fontsize=9)
    ax.tick_params(colors='#9ca3af')
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.grid(alpha=0.1, color='#374151')

axes[0].set_title('', color='white')
axes[-1].set_xlabel('Step', color='#9ca3af', fontsize=10)

# Phase legend
legend_p = [mpatches.Patch(color=phase_colors[i], alpha=0.6, label=phase_labels[i])
             for i in range(3)]
fig.legend(handles=legend_p, loc='lower center', ncol=3,
           facecolor='#1f2937', labelcolor='white', fontsize=9,
           bbox_to_anchor=(0.5, -0.015))

plt.tight_layout()
plt.savefig('telemetry_timeline_36step.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved telemetry_timeline_36step.png")
try:
    from google.colab import files; files.download('telemetry_timeline_36step.png')
except: pass


Saved telemetry_timeline_36step.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 16 — Save LoRA Adapters

In [22]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/orbital_lora_v41'
except Exception:
    SAVE_PATH = '/content/orbital_lora_v41'

os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

summary = {
    'version': '4.1',
    'model':   MODEL_NAME,
    'lora_rank': LORA_RANK,
    'grpo_steps': getattr(grpo_cfg, 'max_steps', getattr(grpo_cfg, 'num_train_epochs', '?')),
    'baseline': {t: float(np.mean(v)) for t, v in baseline.items()},
    'post':     {t: float(np.mean(v)) for t, v in post.items()},
    'improvement': {
        t: f'{(np.mean(post[t])-np.mean(baseline[t]))/np.mean(baseline[t])*100:+.1f}%'
        for t in ['easy', 'medium', 'hard']
    },
    'train_rewards': train_rewards,
}
with open(f'{SAVE_PATH}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved → {SAVE_PATH}')
print('Improvements:')
for t, v in summary['improvement'].items():
    print(f'  {t}: {v}')


Saved → /content/orbital_lora_v41
Improvements:
  easy: +13.2%
  medium: +17.3%
  hard: -3.2%


## Cell 17 — Final Summary

In [23]:
print('=' * 60)
print('ORBITAL ANOMALY OPENENV — v4.1 TRAINING COMPLETE')
print('=' * 60)
print()
print('RESULTS:')
for task in ['easy', 'medium', 'hard']:
    b = np.mean(baseline[task])
    a = np.mean(post[task])
    p = (a - b) / b * 100
    sym = 'IMPROVED ✓' if p > 0.5 else ('FLAT' if abs(p) < 0.5 else 'WORSE')
    print(f'  {task:<8}: {b:.3f} → {a:.3f}  ({p:+.1f}%)  {sym}')

print()
print('v4.1 FIXES:')
print('  Reward fn uses prompt_seed → deterministic state → real group advantage')
print('  Heuristic: eclipse+battery BEFORE comms in priority order')
print('  model.eval() only — no Unsloth kernel swapping')
print()
print('THEMES:')
print('  Theme 3 (World Modeling):  13-fault belief state per step in metadata')
print('  Theme 2 (Long-Horizon):    36-step 3-phase Extended Mission Mode')
print('  Theme 1 (Multi-Agent):     Commander + EPS/Thermal/Comms specialists')
print()
print('NEXT: Write HuggingFace blog post with training_results.png')


ORBITAL ANOMALY OPENENV — v4.1 TRAINING COMPLETE

RESULTS:
  easy    : 0.639 → 0.723  (+13.2%)  IMPROVED ✓
  medium  : 0.680 → 0.798  (+17.3%)  IMPROVED ✓
  hard    : 0.118 → 0.114  (-3.2%)  WORSE

v4.1 FIXES:
  Reward fn uses prompt_seed → deterministic state → real group advantage
  Heuristic: eclipse+battery BEFORE comms in priority order
  model.eval() only — no Unsloth kernel swapping

THEMES:
  Theme 3 (World Modeling):  13-fault belief state per step in metadata
  Theme 2 (Long-Horizon):    36-step 3-phase Extended Mission Mode
  Theme 1 (Multi-Agent):     Commander + EPS/Thermal/Comms specialists

NEXT: Write HuggingFace blog post with training_results.png


## Cell 17b — Final Summary Dashboard

In [24]:
# ── Visual: Final dashboard — all key results in one figure ──────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

BG, PANEL = '#0a0e1a', '#111827'
tasks_lbl  = ['Easy','Medium','Hard']
VALID_ACTIONS = ['rotate_to_sun','disable_payload','reboot_comms',
                 'enter_safe_mode','switch_power_bus','noop']
short_acts = ['rotate','disable','reboot','safe','pwr_bus','noop']

pre_m  = [np.mean(baseline[t.lower()]) for t in tasks_lbl]
post_m = [np.mean(post[t.lower()])     for t in tasks_lbl]
impr   = [(p-b)/max(b,0.001)*100 for b,p in zip(pre_m,post_m)]

fig = plt.figure(figsize=(20, 12), facecolor=BG)
fig.suptitle('Orbital Anomaly OpenEnv — Final Results Dashboard',
             color='white', fontsize=16, fontweight='bold', y=1.01)

gs = plt.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
ax_curve  = fig.add_subplot(gs[0, :2])
ax_bar    = fig.add_subplot(gs[0, 2])
ax_hm_pre = fig.add_subplot(gs[1, 0])
ax_hm_pst = fig.add_subplot(gs[1, 1])
ax_impr   = fig.add_subplot(gs[1, 2])

for ax in [ax_curve, ax_bar, ax_hm_pre, ax_hm_pst, ax_impr]:
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')

# Training curve
if len(train_rewards) >= 3:
    sm = [np.mean(train_rewards[max(0,i-5):i+1]) for i in range(len(train_rewards))]
    ax_curve.fill_between(train_steps, train_rewards, alpha=0.12, color='#6366f1')
    ax_curve.plot(train_steps, train_rewards, color='#374151', alpha=0.35, lw=1)
    ax_curve.plot(train_steps, sm, color='#6366f1', lw=2.5, label='Smoothed reward')
    ax_curve.axhline(pre_m[0], color='#ef4444', ls='--', lw=1.5, label=f'Pre ({pre_m[0]:.3f})')
    ax_curve.axhline(post_m[0], color='#10b981', ls=':', lw=1.5, label=f'Post ({post_m[0]:.3f})')
    ax_curve.legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
else:
    ax_curve.text(0.5, 0.5, "Train model to see reward curve",
                  ha='center', va='center', color='#6b7280', transform=ax_curve.transAxes, fontsize=12)
ax_curve.set_title("GRPO Training Reward Curve", color='white', fontsize=12, fontweight='bold')
ax_curve.set_xlabel("Step", color='#9ca3af'); ax_curve.set_ylabel("Reward", color='#9ca3af')
ax_curve.grid(alpha=0.12, color='#374151')

# Pre/post bar
x, w = np.arange(3), 0.32
b1 = ax_bar.bar(x-w/2, pre_m,  w, color='#ef4444', alpha=0.85, label='Before')
b2 = ax_bar.bar(x+w/2, post_m, w, color='#10b981', alpha=0.85, label='After')
for bars, vals in [(b1,pre_m),(b2,post_m)]:
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x()+bar.get_width()/2, v+0.006,
                    f'{v:.3f}', ha='center', color='white', fontsize=9)
ax_bar.axhline(0.45, color='white', ls=':', lw=1.2, alpha=0.5)
ax_bar.text(2.6, 0.46, '0.45', color='#aaa', fontsize=8)
ax_bar.set_xticks(x); ax_bar.set_xticklabels(tasks_lbl, color='#e5e7eb')
ax_bar.set_title("Pre vs Post", color='white', fontsize=12, fontweight='bold')
ax_bar.legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
ax_bar.grid(axis='y', alpha=0.12, color='#374151')

# Action heatmaps
for ax, data_src, title in [(ax_hm_pre, pre_d, 'Action Policy — BEFORE'),
                              (ax_hm_pst, post_d,'Action Policy — AFTER')]:
    mat = np.array([[data_src.get(t.lower(),{}).get(a,0)
                     for a in VALID_ACTIONS] for t in tasks_lbl])
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0, vmax=0.65)
    ax.set_yticks(range(3)); ax.set_yticklabels(tasks_lbl, color='white', fontsize=10)
    ax.set_xticks(range(6)); ax.set_xticklabels(short_acts, rotation=28, fontsize=9, color='#e5e7eb')
    for i in range(3):
        for j in range(6):
            v = mat[i,j]
            ax.text(j, i, f'{v:.0%}', ha='center', va='center',
                    color='white' if v < 0.35 else 'black', fontsize=9, fontweight='bold')
    ax.set_title(title, color='white', fontsize=11, fontweight='bold')

# Improvement bars
bar_c = ['#10b981' if v >= 0 else '#ef4444' for v in impr]
b_im = ax_impr.bar(tasks_lbl, impr, color=bar_c, alpha=0.85, width=0.45)
for bar, v in zip(b_im, impr):
    ax_impr.text(bar.get_x()+bar.get_width()/2, v+(2 if v>=0 else -5),
                 f'{v:+.1f}%', ha='center', color='white', fontsize=12, fontweight='bold')
ax_impr.axhline(0, color='white', lw=1, alpha=0.35)
ax_impr.set_title("Reward Improvement %", color='white', fontsize=12, fontweight='bold')
ax_impr.set_ylabel("% change vs baseline", color='#9ca3af')
ax_impr.grid(axis='y', alpha=0.12, color='#374151')

plt.savefig('final_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved final_dashboard.png")
try:
    from google.colab import files; files.download('final_dashboard.png')
except: pass


Saved final_dashboard.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>